# Adjustable data chopping

Run all cells, choose a raw or trimmed recording, and click **Load recording**. Set **Parts** to split the recording into that many consecutive parts with equal row counts (differing by at most one row). All columns and original row order are preserved.

Drag the **Cut** sliders to fine-tune the boundaries; a cut is the zero-based row where the next part starts. Each slider is constrained between its neighboring cuts, so parts always retain at least one row. **Reset equal cuts** restores equal sizes. The preview updates after a slider is released. Equal row counts may have different durations if sampling is irregular or intervals were removed.

Click **Export parts** to save CSVs and a boundary manifest under `chopped_data/`. Every export gets a separate folder; source files remain unchanged. Change the selected file and click **Load recording** to switch recordings.


In [ ]:
from pathlib import Path
import math
import os

# Use a project-local cache so Matplotlib does not rebuild its font index.
MPL_CACHE = Path('.cache/matplotlib')
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

DATA_DIR = Path('/Users/aawni/Projects/model/data/trimmed_data')
OUTPUT_DIR = Path('/Users/aawni/Projects/model/data/chopped_data')
MAX_PLOT_POINTS = 2_500

TIME_COLUMN = 'secondsElapsed'
CONTEXT_COLUMNS = ['weight', 'set', 'reps', 'activity', 'activityEncoded']
SENSOR_GROUPS = {
    'Rotation rate': [
        'wristMotion_rotationRateX', 'wristMotion_rotationRateY', 'wristMotion_rotationRateZ'
    ],
    'Gravity': [
        'wristMotion_gravityX', 'wristMotion_gravityY', 'wristMotion_gravityZ'
    ],
    'Acceleration': [
        'wristMotion_accelerationX', 'wristMotion_accelerationY', 'wristMotion_accelerationZ'
    ],
    'Quaternion': [
        'wristMotion_quaternionW', 'wristMotion_quaternionX',
        'wristMotion_quaternionY', 'wristMotion_quaternionZ'
    ],
}
REQUIRED_COLUMNS = [TIME_COLUMN, *sum(SENSOR_GROUPS.values(), []), *CONTEXT_COLUMNS]
COLORS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']

plt.style.use('seaborn-v0_8-whitegrid')

# Split positions are row boundaries: each row belongs to exactly one part.
files = sorted(DATA_DIR.rglob('*.csv'))
if not files:
    raise FileNotFoundError('No raw or trimmed CSV files found.')
file_picker = widgets.Dropdown(options=[(str(p), p) for p in files], description='Recording:', layout=widgets.Layout(width='850px'))
parts = widgets.BoundedIntText(value=5, min=1, max=1000, description='Parts:')
load_button = widgets.Button(description='Load recording', button_style='primary')
equal_button = widgets.Button(description='Reset equal cuts')
export_button = widgets.Button(description='Export parts', button_style='success')
reset_time = widgets.Checkbox(value=False, description='Start each part at time 0')
group_picker = widgets.Dropdown(options=list(SENSOR_GROUPS), description='Sensor:')
cut_controls = widgets.VBox()
plot_output = widgets.Output()
status_output = widgets.Output()
current_df = None
current_path = None
cut_inputs = []
updating_cut_ranges = False

def boundaries():
    cuts = [int(control.value) for control in cut_inputs]
    edges = [0, *cuts, len(current_df)]
    if any(a >= b for a, b in zip(edges, edges[1:])):
        raise ValueError('Cuts must increase strictly, leaving at least one row in each part.')
    return edges

def preview(*_):
    if current_df is None:
        return
    with plot_output:
        clear_output(wait=True)
        try:
            edges = boundaries()
        except ValueError as error:
            print(error)
            return
        summary = []
        for i, (a, b) in enumerate(zip(edges, edges[1:]), 1):
            row = {'part': i, 'first row (0-based)': a, 'last row (inclusive)': b-1, 'rows': b-a}
            if TIME_COLUMN in current_df:
                row.update(start_seconds=current_df[TIME_COLUMN].iloc[a], end_seconds=current_df[TIME_COLUMN].iloc[b-1])
            summary.append(row)
        display(pd.DataFrame(summary))
        columns = [c for c in SENSOR_GROUPS[group_picker.value] if c in current_df]
        fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
        stride = max(1, math.ceil(len(current_df) / MAX_PLOT_POINTS))
        x = np.arange(0, len(current_df), stride)
        for color, column in zip(COLORS, columns):
            ax.plot(x, pd.to_numeric(current_df[column].iloc[::stride], errors='coerce'), label=column.rsplit('_', 1)[-1], color=color, linewidth=.8)
        for i, (a, b) in enumerate(zip(edges, edges[1:])):
            ax.axvspan(a, b, color=COLORS[i % len(COLORS)], alpha=.1)
        for cut in edges[1:-1]:
            ax.axvline(cut, color='black', linestyle='--', linewidth=1)
        ax.set(xlabel='Row position (original order)', ylabel=group_picker.value, title=f'{current_path.name} — {len(edges)-1} parts')
        if columns:
            ax.legend()
        plt.show()
        plt.close(fig)

def update_cut_ranges():
    """Keep each draggable cut between its neighboring cuts."""
    global updating_cut_ranges
    if not cut_inputs:
        return
    updating_cut_ranges = True
    try:
        values = [int(control.value) for control in cut_inputs]
        for i, control in enumerate(cut_inputs):
            lower = 1 if i == 0 else values[i - 1] + 1
            upper = len(current_df) - 1 if i == len(values) - 1 else values[i + 1] - 1
            control.min = lower
            control.max = upper
    finally:
        updating_cut_ranges = False

def cut_changed(change):
    if updating_cut_ranges or current_df is None:
        return
    index = cut_inputs.index(change['owner'])
    values = [int(control.value) for control in cut_inputs]
    lower = 1 if index == 0 else values[index - 1] + 1
    upper = len(current_df) - 1 if index == len(values) - 1 else values[index + 1] - 1
    bounded_value = min(max(int(change['new']), lower), upper)
    if bounded_value != change['new']:
        change['owner'].value = bounded_value
    update_cut_ranges()
    preview()

def equal_cuts(*_):
    global cut_inputs
    if current_df is None:
        return
    count = min(parts.value, len(current_df))
    if parts.value != count:
        parts.value = count
        return
    # Give the first remainder parts one extra row.
    size, remainder = divmod(len(current_df), count)
    cuts = [i * size + min(i, remainder) for i in range(1, count)]
    cut_inputs = [widgets.IntSlider(
        value=cut, min=1, max=len(current_df)-1, description=f'Cut {i}:',
        continuous_update=False, readout=True, readout_format='d',
        style={'description_width': '55px'}, layout=widgets.Layout(width='850px')
    ) for i, cut in enumerate(cuts, 1)]
    for control in cut_inputs:
        control.observe(cut_changed, names='value')
    cut_controls.children = tuple(cut_inputs)
    update_cut_ranges()
    preview()

def load_recording(*_):
    global current_df, current_path
    with status_output:
        clear_output(wait=True)
        try:
            path = Path(file_picker.value)
            frame = pd.read_csv(path, low_memory=False)
            if frame.empty:
                raise ValueError('The selected CSV has no rows.')
            current_df, current_path = frame, path
            equal_cuts()
            print(f'Loaded {len(frame):,} rows. Adjust Parts or edit the cut row positions below.')
        except Exception as error:
            current_df = None
            cut_controls.children = ()
            plot_output.clear_output()
            print(f'Could not load recording: {error}')

def export_parts(*_):
    with status_output:
        clear_output(wait=True)
        if current_df is None:
            print('Load a recording first.')
            return
        try:
            edges = boundaries()
            frames = []
            for a, b in zip(edges, edges[1:]):
                frame = current_df.iloc[a:b].copy()
                if reset_time.value and TIME_COLUMN in frame:
                    elapsed = pd.to_numeric(frame[TIME_COLUMN], errors='coerce')
                    if elapsed.isna().any():
                        raise ValueError('Cannot reset time: secondsElapsed contains missing or invalid values.')
                    frame[TIME_COLUMN] = elapsed - elapsed.iloc[0]
                frames.append(frame)
            # A separate folder per export prevents accidental replacement of previous cuts.
            from datetime import datetime
            folder = OUTPUT_DIR / (current_path.stem + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
            folder.mkdir(parents=True, exist_ok=False)
            manifest = []
            for i, (frame, a, b) in enumerate(zip(frames, edges, edges[1:]), 1):
                name = f'part_{i:03d}.csv'
                frame.to_csv(folder / name, index=False)
                manifest.append({'file': name, 'start_row': a, 'stop_row_exclusive': b, 'rows': len(frame)})
            pd.DataFrame(manifest).to_csv(folder / 'manifest.csv', index=False)
            print(f'Saved {len(frames)} parts ({len(current_df):,} total rows) to {folder}')
        except Exception as error:
            print(f'Export failed: {error}')

load_button.on_click(load_recording)
equal_button.on_click(equal_cuts)
parts.observe(equal_cuts, names='value')
group_picker.observe(preview, names='value')
export_button.on_click(export_parts)
display(widgets.VBox([file_picker, load_button, widgets.HBox([parts, equal_button, group_picker]), cut_controls, widgets.HBox([reset_time, export_button]), status_output]), plot_output)
load_recording()
